In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# DengueTect — Data Processing Pipeline\n",
    "**Group 10 | EARIST**\n",
    "\n",
    "This notebook handles:\n",
    "- Loading raw dengue case records from Marikina CHO\n",
    "- Filtering for target barangays: Nangka, Tumana, Malanday\n",
    "- Aggregating individual records into weekly case counts\n",
    "- Fetching climate data from Open-Meteo API\n",
    "- Applying 4-week lag to climate variables\n",
    "- Exporting the final merged dataset for model training"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Imports\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import requests\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "print('Libraries loaded.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Config\n",
    "TARGET_BARANGAYS = ['Nangka', 'Tumana', 'Malanday']\n",
    "\n",
    "DATA_FILES = [\n",
    "    '../data/raw/dengue-cases-2022.xlsx',\n",
    "    '../data/raw/dengue-cases-2023.xlsx',\n",
    "    '../data/raw/dengue-cases-2024.xlsx',\n",
    "    '../data/raw/dengue-cases-2025.xlsx',\n",
    "    '../data/raw/dengue-cases-2026.xlsx',\n",
    "]\n",
    "\n",
    "BARANGAY_COL  = '(Current Address) Barangay'\n",
    "DATE_COL      = 'DOnset'\n",
    "\n",
    "print('Config set.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load and Combine Raw Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dfs = []\n",
    "for f in DATA_FILES:\n",
    "    df = pd.read_excel(f)\n",
    "    dfs.append(df)\n",
    "    print(f'Loaded {f} — {len(df)} rows')\n",
    "\n",
    "raw = pd.concat(dfs, ignore_index=True)\n",
    "print(f'\\nTotal rows: {len(raw)}')\n",
    "print(f'Columns: {list(raw.columns[:6])}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Filter Target Barangays"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Filter for Nangka, Tumana, Malanday only\n",
    "filtered = raw[raw[BARANGAY_COL].isin(TARGET_BARANGAYS)].copy()\n",
    "filtered = filtered.reset_index(drop=True)\n",
    "\n",
    "print(f'Filtered rows: {len(filtered)}')\n",
    "print(filtered[BARANGAY_COL].value_counts())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Keep only relevant columns\n",
    "cols_keep = [BARANGAY_COL, DATE_COL, 'Sex', 'AgeYears', 'ClinClass', 'Case Classification', 'outcome']\n",
    "filtered = filtered[cols_keep]\n",
    "filtered.columns = ['barangay', 'date_onset', 'sex', 'age', 'clin_class', 'case_class', 'outcome']\n",
    "\n",
    "print(filtered.head())\n",
    "print(f'\\nDate column dtype: {filtered[\"date_onset\"].dtype}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Parse Dates\n",
    "Excel stores dates as serial numbers — need to convert."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Convert Excel serial dates to datetime\n",
    "def parse_excel_date(val):\n",
    "    if pd.isnull(val):\n",
    "        return pd.NaT\n",
    "    if isinstance(val, (int, float)):\n",
    "        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(val))\n",
    "    return pd.to_datetime(val, errors='coerce')\n",
    "\n",
    "filtered['date_onset'] = filtered['date_onset'].apply(parse_excel_date)\n",
    "filtered = filtered.dropna(subset=['date_onset'])\n",
    "\n",
    "print(f'Date range: {filtered[\"date_onset\"].min()} to {filtered[\"date_onset\"].max()}')\n",
    "print(f'Rows after date parse: {len(filtered)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Weekly Aggregation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Add ISO week and year\n",
    "filtered['week'] = filtered['date_onset'].dt.to_period('W')\n",
    "\n",
    "# Aggregate: count cases per barangay per week\n",
    "weekly = (\n",
    "    filtered\n",
    "    .groupby(['barangay', 'week'])\n",
    "    .size()\n",
    "    .reset_index(name='case_count')\n",
    ")\n",
    "\n",
    "weekly['week_start'] = weekly['week'].apply(lambda w: w.start_time)\n",
    "weekly = weekly.sort_values(['barangay', 'week_start']).reset_index(drop=True)\n",
    "\n",
    "print(f'Weekly records: {len(weekly)}')\n",
    "print(weekly.head(10))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary per barangay\n",
    "print(weekly.groupby('barangay')['case_count'].describe())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Climate Data — Open-Meteo API\n",
    "Fetch weekly rainfall, temperature, humidity for Marikina City coordinates."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Marikina City coordinates\n",
    "LAT = 14.6507\n",
    "LON = 121.1029\n",
    "\n",
    "# Date range\n",
    "START_DATE = '2022-01-01'\n",
    "END_DATE   = '2026-03-31'\n",
    "\n",
    "url = (\n",
    "    f'https://archive-api.open-meteo.com/v1/archive'\n",
    "    f'?latitude={LAT}&longitude={LON}'\n",
    "    f'&start_date={START_DATE}&end_date={END_DATE}'\n",
    "    f'&daily=precipitation_sum,temperature_2m_mean,relative_humidity_2m_mean'\n",
    "    f'&timezone=Asia/Manila'\n",
    ")\n",
    "\n",
    "print('Fetching climate data...')\n",
    "resp = requests.get(url)\n",
    "data = resp.json()\n",
    "\n",
    "climate = pd.DataFrame({\n",
    "    'date':     pd.to_datetime(data['daily']['time']),\n",
    "    'rainfall': data['daily']['precipitation_sum'],\n",
    "    'temp':     data['daily']['temperature_2m_mean'],\n",
    "    'humidity': data['daily']['relative_humidity_2m_mean'],\n",
    "})\n",
    "\n",
    "print(f'Climate rows: {len(climate)}')\n",
    "print(climate.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Aggregate climate to weekly\n",
    "climate['week'] = climate['date'].dt.to_period('W')\n",
    "\n",
    "climate_weekly = (\n",
    "    climate\n",
    "    .groupby('week')\n",
    "    .agg(rainfall=('rainfall','sum'), temp=('temp','mean'), humidity=('humidity','mean'))\n",
    "    .reset_index()\n",
    ")\n",
    "\n",
    "climate_weekly['week_start'] = climate_weekly['week'].apply(lambda w: w.start_time)\n",
    "print(f'Weekly climate rows: {len(climate_weekly)}')\n",
    "print(climate_weekly.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Apply 4-Week Lag to Climate Variables"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Merge weekly cases with climate data\n",
    "merged = weekly.merge(climate_weekly[['week','rainfall','temp','humidity']], on='week', how='left')\n",
    "\n",
    "# Apply 4-week lag per barangay\n",
    "lag_weeks = 4\n",
    "lagged_dfs = []\n",
    "\n",
    "for brgy in TARGET_BARANGAYS:\n",
    "    df_brgy = merged[merged['barangay'] == brgy].copy().sort_values('week_start')\n",
    "    df_brgy['rainfall_lag'] = df_brgy['rainfall'].shift(lag_weeks)\n",
    "    df_brgy['temp_lag']     = df_brgy['temp'].shift(lag_weeks)\n",
    "    df_brgy['humidity_lag'] = df_brgy['humidity'].shift(lag_weeks)\n",
    "    lagged_dfs.append(df_brgy)\n",
    "\n",
    "final = pd.concat(lagged_dfs).dropna().reset_index(drop=True)\n",
    "\n",
    "print(f'Final dataset rows: {len(final)}')\n",
    "print(final.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Add outbreak label for Logistic Regression\n",
    "# Threshold: weeks with case_count >= 75th percentile per barangay = outbreak\n",
    "threshold = final.groupby('barangay')['case_count'].transform(lambda x: x.quantile(0.75))\n",
    "final['outbreak'] = (final['case_count'] >= threshold).astype(int)\n",
    "\n",
    "print(final['outbreak'].value_counts())\n",
    "print(final[['barangay','week_start','case_count','outbreak']].head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Export"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Export to CSV\n",
    "final.to_csv('../data/processed_dataset.csv', index=False)\n",
    "print('Exported to data/processed_dataset.csv')\n",
    "print(final.shape)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.14.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}